# P5a0: Colab T4 toolchain trial
先頭の CODE_COMMIT だけを、実装を含む固定コミットの40桁ハッシュに変更してください。
ランタイムをT4にして上から実行します。追加パッケージはインストールしません。
入力の束は手元の bundle コマンドで作った同じファイルをアップロードします。
これは輸送計算ではありません。


In [ ]:
CODE_COMMIT = "931dcbd7ba91a51ee345c011376366883aaf1420"
import sys, subprocess, shutil, platform
print("Python", platform.python_version())
for command in (["nvidia-smi"], ["g++", "--version"]):
    if shutil.which(command[0]) is None:
        raise RuntimeError(command[0] + " がありません。ここで停止します。")
    subprocess.run(command, check=True)
try:
    import cupy as cp
except ImportError as exc:
    raise RuntimeError("CuPy がありません。ここで停止します。") from exc
print("CuPy", cp.__version__)
print("CUDA runtime", cp.cuda.runtime.runtimeGetVersion())
print("CUDA driver", cp.cuda.runtime.driverGetVersion())
print("GPU", cp.cuda.runtime.getDeviceProperties(0)["name"])


In [ ]:
from pathlib import Path
import os, re
if not re.fullmatch(r"[0-9a-f]{40}", CODE_COMMIT):
    raise ValueError("先頭の CODE_COMMIT を実装の固定コミットにしてください。")
REPO = Path.home() / "p5a0_checkout"
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/gakugaku3333/chatcarlo.git", str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_COMMIT], check=True)
actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert actual == CODE_COMMIT, (actual, CODE_COMMIT)
os.chdir(REPO)
print("Code commit", actual)
WORK = Path.home() / "p5a0_work"
WORK.mkdir(exist_ok=True)


In [ ]:
from google.colab import files
import hashlib
os.chdir(WORK)
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("入力の npz ファイルを1つだけ選んでください。")
name, content = next(iter(uploaded.items()))
if not name.endswith(".npz"):
    raise ValueError("npz が必要です。")
BUNDLE = WORK / "p5a0_inputs.npz"
BUNDLE.write_bytes(content)
print("Bundle SHA-256", hashlib.sha256(content).hexdigest())
os.chdir(REPO)


In [ ]:
RESULT = WORK / "p5a0_colab.json"
env = dict(os.environ, PYTHONPATH=str(REPO))
completed = subprocess.run([sys.executable, "-m", "prototypes.mv_electron.gpu.p5a0", "colab",
    "--bundle", str(BUNDLE), "--work", str(WORK), "--out", str(RESULT)], env=env,
    capture_output=True, text=True)
print(completed.stdout)
print(completed.stderr)
print("Exit status", completed.returncode)
# A failed check still produces JSON; download it for diagnosis.


In [ ]:
import json
result = json.loads(RESULT.read_text())
print(json.dumps({key: result[key] for key in ("verdict", "environment", "checks")}, indent=2, ensure_ascii=False))
files.download(str(RESULT))
